**Cloning the PlantDoc Repository**

In [ ]:
import os

REPO_URL = "https://github.com/pratikkayal/PlantDoc-Dataset.git"
# Add a specific subfolder name at the end
WORKDIR = "/content/drive/MyDrive/Colab Notebooks/Final Project folder/PlantDoc-Dataset"

if not os.path.exists(WORKDIR):
    # Use quotes around "{WORKDIR}" to safely handle spaces in folder names
    !git clone {REPO_URL} "{WORKDIR}"
    print("Repository cloned successfully.")
else:
    print("Repository folder already exists.")

Cloning into '/content/drive/MyDrive/Colab Notebooks/Final Project folder/PlantDoc-Dataset'...
remote: Enumerating objects: 2670, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 2670 (delta 22), reused 22 (delta 22), pack-reused 2635 (from 1)
Receiving objects: 100% (2670/2670), 932.92 MiB | 16.68 MiB/s, done.
Resolving deltas: 100% (24/24), done.
Updating files: 100% (2581/2581), done.
Repository cloned successfully.


**Generating a Count of the Available Tomato subset images**

In [ ]:
import os

# Base directory where your PlantDoc repository is cloned
WORKDIR = "/content/drive/MyDrive/Colab Notebooks/Final Project folder/PlantDoc-Dataset"

# The specific 5 tomato folders to count
TARGET_FOLDERS = [
    "Tomato Early blight leaf",
    "Tomato leaf",
    "Tomato leaf mosaic virus",
    "Tomato leaf yellow virus",
    "Tomato Septoria leaf spot"
]

def count_tomato_images(base_dir):
    print(f"Checking dataset folder: {base_dir}\n" + "="*50)

    # Supported image file extensions
    valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp')

    total_images_all_splits = 0

    for split in ['train', 'test']:
        split_path = os.path.join(base_dir, split)

        # Check uppercase fallback ('TRAIN' / 'TEST') if lowercase isn't present
        if not os.path.exists(split_path):
            split_path = os.path.join(base_dir, split.upper())

        if not os.path.exists(split_path):
            print(f"\n[!] Could not find directory for '{split}' split.")
            continue

        print(f"\n📁 SPLIT: {split.upper()}")
        split_total = 0

        for folder_name in TARGET_FOLDERS:
            folder_path = os.path.join(split_path, folder_name)

            if os.path.exists(folder_path):
                # Count files that are valid images
                file_count = len([
                    f for f in os.listdir(folder_path)
                    if f.lower().endswith(valid_extensions)
                ])
                print(f"  • {folder_name:<30} : {file_count:>4} images")
                split_total += file_count
            else:
                print(f"  • {folder_name:<30} : [NOT FOUND]")

        print(f"  --------------------------------------------------")
        print(f"  Subtotal for {split.upper()}: {split_total} images")
        total_images_all_splits += split_total

    print("\n" + "="*50)
    print(f"TOTAL IMAGES ACROSS ALL 5 CLASSES: {total_images_all_splits}")

# Run the counter
count_tomato_images(WORKDIR)

Checking dataset folder: /content/drive/MyDrive/Colab Notebooks/Final Project folder/PlantDoc-Dataset

📁 SPLIT: TRAIN
  • Tomato Early blight leaf       :   79 images
  • Tomato leaf                    :   55 images
  • Tomato leaf mosaic virus       :   44 images
  • Tomato leaf yellow virus       :   70 images
  • Tomato Septoria leaf spot      :  140 images
  --------------------------------------------------
  Subtotal for TRAIN: 388 images

📁 SPLIT: TEST
  • Tomato Early blight leaf       :    9 images
  • Tomato leaf                    :    8 images
  • Tomato leaf mosaic virus       :   10 images
  • Tomato leaf yellow virus       :    6 images
  • Tomato Septoria leaf spot      :   11 images
  --------------------------------------------------
  Subtotal for TEST: 44 images

TOTAL IMAGES ACROSS ALL 5 CLASSES: 432


**PyTorch Transforms, DataLoaders, & Weighted Loss Setup**

In [ ]:
import os
import shutil
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# ---------------------------------------------------------
# 1. PATHS SETUP
# ---------------------------------------------------------
SOURCE_BASE = "/content/drive/MyDrive/Colab Notebooks/Final Project folder/PlantDoc-Dataset"
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/Final Project folder/Tomato_Dataset_Split"

TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR = os.path.join(DATA_DIR, "val")
TEST_DIR = os.path.join(DATA_DIR, "test")

CLASS_MAP = {
    "Tomato Early blight leaf": "Early_Blight",
    "Tomato Septoria leaf spot": "Septoria",
    "Tomato leaf mosaic virus": "Mosaic",
    "Tomato leaf yellow virus": "Yellow_Virus",
    "Tomato leaf": "Healthy"
}

# ---------------------------------------------------------
# 2. CREATE SPLIT DIRECTORIES IF THEY DO NOT EXIST
# ---------------------------------------------------------
if not os.path.exists(TRAIN_DIR):
    print("Split directories not found. Building dataset split now...")

    all_filepaths = []
    all_labels = []
    valid_exts = ('.jpg', '.jpeg', '.png', '.bmp')

    # Collect all image filepaths
    for original_folder, clean_label in CLASS_MAP.items():
        for split in ['train', 'test', 'TRAIN', 'TEST']:
            dir_path = os.path.join(SOURCE_BASE, split, original_folder)
            if os.path.exists(dir_path):
                for fname in os.listdir(dir_path):
                    if fname.lower().endswith(valid_exts):
                        all_filepaths.append(os.path.join(dir_path, fname))
                        all_labels.append(clean_label)

    # Perform Stratified 80/10/10 Split
    X_train, X_temp, y_train, y_temp = train_test_split(
        all_filepaths, all_labels, test_size=0.20, random_state=42, stratify=all_labels
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
    )

    # Copy files into structured split folders
    splits_data = {
        'train': (X_train, y_train),
        'val': (X_val, y_val),
        'test': (X_test, y_test)
    }

    for split_name, (files, labels) in splits_data.items():
        for filepath, label in zip(files, labels):
            dest_dir = os.path.join(DATA_DIR, split_name, label)
            os.makedirs(dest_dir, exist_ok=True)
            shutil.copy2(filepath, os.path.join(dest_dir, os.path.basename(filepath)))

    print("Stratified dataset split completed successfully!")
else:
    print("Found existing split directory at:", DATA_DIR)

# ---------------------------------------------------------
# 3. PYTORCH TRANSFORMS & DATALOADERS
# ---------------------------------------------------------
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 16

# Training Augmentations (On-the-fly)
train_transforms = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# Deterministic Transforms for Validation & Testing
val_test_transforms = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# Load ImageFolder Datasets
train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transforms)
val_dataset = datasets.ImageFolder(root=VAL_DIR, transform=val_test_transforms)
test_dataset = datasets.ImageFolder(root=TEST_DIR, transform=val_test_transforms)

# Build DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# ---------------------------------------------------------
# 4. COMPUTE LOSS WEIGHTS FOR CLASS IMBALANCE
# ---------------------------------------------------------
class_names = train_dataset.classes
train_labels = train_dataset.targets

computed_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_weights_tensor = torch.tensor(computed_weights, dtype=torch.float).to(device)

print(f"\nTarget Classes ({len(class_names)}): {class_names}")
print("Computed Class Weights:")
for cls, weight in zip(class_names, class_weights_tensor):
    print(f"  • {cls:<15}: {weight.item():.4f}")

# Loss function criterion
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

print(f"\nSetup complete on device '{device}'.")
print(f"Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)} | Test samples: {len(test_dataset)}")

Split directories not found. Building dataset split now...
Stratified dataset split completed successfully!

Target Classes (5): ['Early_Blight', 'Healthy', 'Mosaic', 'Septoria', 'Yellow_Virus']
Computed Class Weights:
  • Early_Blight   : 0.9800
  • Healthy        : 1.3720
  • Mosaic         : 1.5953
  • Septoria       : 0.5717
  • Yellow_Virus   : 1.1433

Setup complete on device 'cuda'.
Train samples: 343 | Val samples: 43 | Test samples: 44


Checking for duplicates, missing, or corrupted files.

In [ ]:
from collections import Counter

# 1. Target subfolders to inspect inside PlantDoc-Dataset
TARGET_FOLDERS = [
    "Tomato Early blight leaf",
    "Tomato Septoria leaf spot",
    "Tomato leaf mosaic virus",
    "Tomato leaf yellow virus",
    "Tomato leaf"
]

# 2. Extract all filenames indexed by PyTorch across all datasets
pytorch_indexed_files = set()
for dataset in [train_dataset, val_dataset, test_dataset]:
    for path, _ in dataset.samples:
        pytorch_indexed_files.add(os.path.basename(path))

# 3. Scan source folders, track duplicate filenames, and find missing files
all_source_filenames = []
file_paths_map = {}
missing_files = []

for split in ['train', 'test', 'TRAIN', 'TEST']:
    for folder in TARGET_FOLDERS:
        target_path = os.path.join(SOURCE_BASE, split, folder)
        if os.path.exists(target_path):
            for f in os.listdir(target_path):
                if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                    all_source_filenames.append(f)
                    file_paths_map.setdefault(f, []).append(os.path.join(target_path, f))

                    if f not in pytorch_indexed_files:
                        missing_files.append(os.path.join(target_path, f))

# 4. Count duplicates
filename_counts = Counter(all_source_filenames)
duplicate_filenames = {name: count for name, count in filename_counts.items() if count > 1}

print("=" * 60)
print(f"Total raw image files found in target folders : {len(all_source_filenames)}")
print(f"Total unique filenames                      : {len(filename_counts)}")
print(f"Total duplicate instances (overwritten)     : {len(all_source_filenames) - len(filename_counts)}")
print("=" * 60)

if duplicate_filenames:
    print(f"\nFound {len(duplicate_filenames)} duplicate filename(s):")
    for fname, count in duplicate_filenames.items():
        print(f"\n  • Filename: '{fname}' (Appears {count} times)")
        for path in file_paths_map[fname]:
            print(f"     - Path: {path}")
else:
    print("\nNo duplicate filenames detected.")

print("\n" + "=" * 60)
print(f"Unindexed by PyTorch: {len(missing_files)} files")

Total raw image files found in target folders : 432
Total unique filenames                      : 430
Total duplicate instances (overwritten)     : 2

Found 2 duplicate filename(s):

  • Filename: 'early-blight-septoria-ls-fig-3.jpg' (Appears 2 times)
     - Path: /content/drive/MyDrive/Colab Notebooks/Final Project folder/PlantDoc-Dataset/train/Tomato Septoria leaf spot/early-blight-septoria-ls-fig-3.jpg
     - Path: /content/drive/MyDrive/Colab Notebooks/Final Project folder/PlantDoc-Dataset/test/Tomato Septoria leaf spot/early-blight-septoria-ls-fig-3.jpg

  • Filename: 'tylcv-seminar-1-638.jpg' (Appears 2 times)
     - Path: /content/drive/MyDrive/Colab Notebooks/Final Project folder/PlantDoc-Dataset/train/Tomato leaf yellow virus/tylcv-seminar-1-638.jpg
     - Path: /content/drive/MyDrive/Colab Notebooks/Final Project folder/PlantDoc-Dataset/test/Tomato leaf yellow virus/tylcv-seminar-1-638.jpg

Unindexed by PyTorch: 0 files


**Generating a Final Count before training begings**

In [ ]:
import os

# Base directory where your split dataset is saved
SPLIT_DIR = "/content/drive/MyDrive/Colab Notebooks/Final Project folder/Tomato_Dataset_Split"

# The 5 standardized class folder names inside the split directory
TARGET_CLASSES = [
    "Early_Blight",
    "Healthy",
    "Mosaic",
    "Septoria",
    "Yellow_Virus"
]

def count_split_images(base_dir):
    print(f"Checking dataset folder: {base_dir}\n" + "="*50)

    valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp')
    total_images_all_splits = 0

    # Iterate through the three dataset splits
    for split in ['train', 'val', 'test']:
        split_path = os.path.join(base_dir, split)

        if not os.path.exists(split_path):
            print(f"\n[!] Could not find directory for '{split}' split.")
            continue

        print(f"\n📁 SPLIT: {split.upper()}")
        split_total = 0

        for class_name in TARGET_CLASSES:
            class_path = os.path.join(split_path, class_name)

            if os.path.exists(class_path):
                file_count = len([
                    f for f in os.listdir(class_path)
                    if f.lower().endswith(valid_extensions)
                ])
                print(f"  • {class_name:<20} : {file_count:>4} images")
                split_total += file_count
            else:
                print(f"  • {class_name:<20} : [NOT FOUND]")

        print(f"  --------------------------------------------------")
        print(f"  Subtotal for {split.upper()}: {split_total} images")
        total_images_all_splits += split_total

    print("\n" + "="*50)
    print(f"TOTAL IMAGES ACROSS ALL 5 CLASSES: {total_images_all_splits}")

# Run the counter
count_split_images(SPLIT_DIR)

Checking dataset folder: /content/drive/MyDrive/Colab Notebooks/Final Project folder/Tomato_Dataset_Split

📁 SPLIT: TRAIN
  • Early_Blight         :   70 images
  • Healthy              :   50 images
  • Mosaic               :   43 images
  • Septoria             :  120 images
  • Yellow_Virus         :   60 images
  --------------------------------------------------
  Subtotal for TRAIN: 343 images

📁 SPLIT: VAL
  • Early_Blight         :    9 images
  • Healthy              :    6 images
  • Mosaic               :    6 images
  • Septoria             :   15 images
  • Yellow_Virus         :    7 images
  --------------------------------------------------
  Subtotal for VAL: 43 images

📁 SPLIT: TEST
  • Early_Blight         :    9 images
  • Healthy              :    7 images
  • Mosaic               :    5 images
  • Septoria             :   15 images
  • Yellow_Virus         :    8 images
  --------------------------------------------------
  Subtotal for TEST: 44 images

TOTAL IMAG

**Fine-Tuning the ResNet50 Baseline Model**

In [ ]:
import os
import time
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

# 1. Hardware Check (Ensure CUDA is active)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training ResNet50 baseline on device: {device}")
if device.type == 'cpu':
    print("⚠️ WARNING: Running on CPU! Change runtime type in Colab: Runtime -> Change runtime type -> T4 GPU")

# 2. Define Class Names safely
try:
    class_names = train_dataset.classes
except NameError:
    class_names = ['Early_Blight', 'Healthy', 'Mosaic', 'Septoria', 'Yellow_Virus']

print(f"Target classes ({len(class_names)}): {class_names}")

# 3. Load Pre-trained ResNet50 Model
resnet_model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# Modify final linear layer to output 5 disease classes
num_ftrs = resnet_model.fc.in_features
resnet_model.fc = nn.Linear(num_ftrs, len(class_names))
resnet_model = resnet_model.to(device)

# 4. Define Loss, Optimizer, and Learning Rate Scheduler
optimizer = optim.AdamW(resnet_model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

# Save directory in Google Drive
MODEL_SAVE_DIR = "/content/drive/MyDrive/Colab Notebooks/Final Project folder/saved_models"
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
BEST_MODEL_PATH = os.path.join(MODEL_SAVE_DIR, "resnet50_baseline_best.pth")

# 5. Training and Validation Function
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=25):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_val_loss = float('inf')

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print("-" * 35)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
                dataloader = train_loader
            else:
                model.eval()
                dataloader = val_loader

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / len(dataloader.dataset)
            epoch_acc = (running_corrects.double() / len(dataloader.dataset)).item() * 100

            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc)

            print(f"{phase.capitalize():<5} Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.2f}%")

            # LR schedule and model saving during validation
            if phase == 'val':
                scheduler.step(epoch_loss)
                if epoch_loss < best_val_loss:
                    best_val_loss = epoch_loss
                    best_model_wts = copy.deepcopy(model.state_dict())
                    torch.save(model.state_dict(), BEST_MODEL_PATH)
                    print(f" --> Best model saved to Drive (Val Loss: {best_val_loss:.4f})")

    time_elapsed = time.time() - since
    print(f"\nTraining complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s")
    print(f"Best Validation Loss: {best_val_loss:.4f}")

    # Load best weights before returning
    model.load_state_dict(best_model_wts)
    return model, history

# 6. Execute Training
trained_resnet, resnet_history = train_model(
    resnet_model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=25
)

Training ResNet50 baseline on device: cpu
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 93.7MB/s]


NameError: name 'class_names' is not defined